# Prepare the features

- BOP features arrangement and averagings
- Library Features arrangement ad averaging
- Matminer features cleanup (decide min, max or avg)
- clear feature list from correlations
- outlier detection based on features
- Feature Distribution plots

 - input : features dataframes from prev steps
 - output: clean features dataframes into picles

In [1]:
import pandas as pd
import seaborn
import matplotlib.pyplot as plt
import copy
import os
import numpy as np
import pdb
from tqdm.auto import tqdm 
dataset = 'Cr-Co-W/'

# Load Features 

In [2]:
descriptorlocation = os.path.join(dataset, 'Descriptors')
PyscalFeaturesFile = os.path.join(descriptorlocation, 'pyscal_steinhardt.kpl')
BopFeaturesFile = os.path.join(descriptorlocation, 'CrCoW_initial_canonical_table_WUBIND_16.pkl')
AtomicFeaturesFile = os.path.join(descriptorlocation, 'matminer_atomic_features.pkl')
CompositionFeaturesFile = os.path.join(descriptorlocation, 'matminer_composition_features.pkl')

In [3]:
PyscalFeatures = pd.read_pickle(PyscalFeaturesFile)

In [4]:
BopFeatures = pd.read_pickle(BopFeaturesFile)

In [5]:
AtomicFeatures = pd.read_pickle(AtomicFeaturesFile)

In [6]:
CompositionFeatures = pd.read_pickle(CompositionFeaturesFile)

## Averaging

In [7]:
neighbours  = {'CN12': 12, 'CN14': 14, 'CN15': 15, 'CN16': 16}

In [8]:
def cn_average(steinhardt, coordination): # *args): #iterable, coordinations, axis=1):
    average = {}
    index, array = steinhardt
    _, coord = coordination
    natoms = len(array)
    for i, atomarray in enumerate(array):
        for polyhedra, nneighbours in neighbours.items():
            average[f'{str(i)}_{polyhedra}'] = np.array(atomarray)[np.array(coord) == nneighbours].sum()/natoms
#    return average
    return  {index: average}

In [9]:
def cnaverage_dataframe(_Features, colids, _Coordinations, nworkers = 3):
    result = {}
    for colid in colids:
        print(f'cnaverage {colid}')
        progress = tqdm(zip(_Features[colid].iteritems(), _Coordinations.iteritems()), total=_Features.shape[0])
        thisresult = [cn_average(steinhardts, coordinations) for steinhardts, coordinations in progress]
    # I prepared this for process_map for parallell calc, but not sure if this is worth it. averaging is fast enaug!
    #        process_map(featurizer, _Features['pyscal_steinhardt'].iteritems(), _Features['pyscal_cn'].iteritems(), max_workers=nworkers)# for featurizer in featurizers}
        thisresult =  dict(map(dict.popitem, thisresult) )
        result.update({colid: pd.DataFrame.from_dict(thisresult,orient='index' )})
    return  result

In [10]:
cn_average((PyscalFeatures.index[0], PyscalFeatures.pyscal_steinhardt.iloc[0]), ('', PyscalFeatures.pyscal_cn.iloc[0]))

{'Co_pv6W_sv6.C14-BBA.FM': {'0_CN12': 0.12440826525579637,
  '0_CN14': 0.0,
  '0_CN15': 0.0,
  '0_CN16': 0.006116135760671434,
  '1_CN12': 2.6167554193863327,
  '1_CN14': 0.0,
  '1_CN15': 0.0,
  '1_CN16': 0.3867610747010618}}

In [16]:
PyscalFeaturesAveraged = cnaverage_dataframe(PyscalFeatures, ['pyscal_steinhardt'], PyscalFeatures.pyscal_cn)

cnaverage pyscal_steinhardt


  0%|          | 0/1682 [00:00<?, ?it/s]

In [23]:
columns = PyscalFeaturesAveraged['pyscal_steinhardt'].columns

In [24]:
NamedFeatures = [ f'pyscal_steinhardt_{column}' for column in columns ]

In [26]:
PyscalFeaturesAveraged['pyscal_steinhardt'].columns = NamedFeatures

# Compositional Features 